In [ ]:
import cv2
import numpy as np
import os
# Класс для предобработки
class VideoPreprocessor:
    def __init__(self, video_path: str):
        self.video_path = video_path
        self.cap = cv2.VideoCapture(video_path)
        
        if not self.cap.isOpened():
            raise ValueError(f"Не удалось открыть видео: {video_path}")
        
        # Вытаскиваем метаданные
        self.fps = self.cap.get(cv2.CAP_PROP_FPS) # fps
        self.total_frames = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT)) # Количество кадров
        self.duration_ms = int((self.total_frames / self.fps) * 1000) # Продолжитеьлность
        
        # Параметры
        self.window_size_ms = 1000  # 1 секунда
        self.sharpness_threshold = 100.0 # Порог резкости
        self.rotation_code = cv2.ROTATE_90_COUNTERCLOCKWISE # Переворачиваем на 90  градусов
    
    def calculate_sharpness(self, frame):
        # Для каждого кадра считаем резкость по метрике лапласиана
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        laplacian = cv2.Laplacian(gray, cv2.CV_64F)
        return laplacian.var() # Возращаем дисперсию
    
    def enhance_frame(self, frame):
        # Считаем CLAHE лдя каждого кадра, чтобы повыстить контрастность
        # lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB) # lab для того, чтобы не трогать цвета, а только яркость
        # clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        # lab[:,:,0] = clahe.apply(lab[:,:,0])
        # frame = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
        
        # # Осветляем тени
        # gamma = 0.9
        # inv_gamma = 1.0 / gamma
        # table = np.array([((i / 255.0) ** (1.0 / inv_gamma) * 255)
        #                   for i in np.arange(0, 256)]).astype("uint8")
        # frame = cv2.LUT(frame, table)
        
        # # Фильтр для удалпения шума
        # frame = cv2.bilateralFilter(frame, d=5, sigmaColor=50, sigmaSpace=50)

    # def enhance_frame_adaptive(frame):
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB) # lab для того, чтобы не трогать цвета, а только яркость

        L = lab[:,:,0].astype(float)
        
        # Оцениваем долю засветов (пиксели > 240)
        glare_ratio = np.sum(L > 240) / L.size
        
        # Адаптивные параметры
        if glare_ratio > 0.15:  # >15% кадра в бликах
            clip_limit = 1.5    # мягче CLAHE
            gamma = 1.1         # чуть затемняем, чтобы вытянуть детали из пересвета
        else:
            clip_limit = 2.0
            gamma = 0.9
            
        # Подавление бликов (простое сжатие ярких пикселей)
        L = np.where(L > 230, L * 0.85, L)
        L = np.clip(L, 0, 255).astype(np.uint8)
        
        # CLAHE
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8,8))
        lab[:,:,0] = clahe.apply(L)
        
        # Осветляем тени
        inv_gamma = 1.0 / gamma
        table = np.array([((i / 255.0) ** (1.0 / inv_gamma) * 255)
                        for i in np.arange(0, 256)]).astype("uint8")
        lab[:,:,0] = cv2.LUT(lab[:,:,0], table)
        
        # Фильтр для удалпения шума
        enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
        return cv2.bilateralFilter(enhanced, d=5, sigmaColor=50, sigmaSpace=50)


        

    
    # Пайплайн предобработки 
    def process(self):

            # Пайплайн такой: читаем видео, поварачиваем, разиваем на окна, в каждом окне выбираем лучший кадр, улучшаем его, возращаем список (timestamp, frame, sharpness)

            best_frames = [] # Лучшие кадры
            buffer = []  # Кадры в текущем окне
            last_flush_ts = 0
            
            frame_idx = 0
            
            while self.cap.isOpened():
                ret, frame = self.cap.read()
                if not ret:
                    break
                
                # Поворот
                frame = cv2.rotate(frame, self.rotation_code)
                
                # Timestamp
                ts_ms = int(self.cap.get(cv2.CAP_PROP_POS_MSEC))
                
                # Sharpness
                sharpness = self.calculate_sharpness(frame)
                
                # Фильтруем размытые
                if sharpness > self.sharpness_threshold:
                    buffer.append((ts_ms, sharpness, frame.copy()))
                
                # Выбираем лучшее в каждом окне
                if ts_ms - last_flush_ts >= self.window_size_ms:
                    if buffer:
                        # Лучший по sharpness
                        best_ts, best_score, best_img = max(buffer, key=lambda x: x[1])
                        
                        # Улучшаем
                        enhanced = self.enhance_frame(best_img)
                        best_frames.append((best_ts, enhanced, best_score))
                    
                    # Сброс
                    buffer = []
                    last_flush_ts = ts_ms
                
                frame_idx += 1
            
            # Последний кадр
            self.cap.release()
            
            return best_frames
    
    def save_frames(self, frames: list, output_dir: str):
        """Сохраняем кадры для отладки"""
        os.makedirs(output_dir, exist_ok=True)
        
        for i, (ts, frame, score) in enumerate(frames):
            filename = f"frame_{ts}_score_{score:.1f}.jpg"
            filepath = os.path.join(output_dir, filename)
            cv2.imwrite(filepath, frame)
        
        print(f"Сохранено {len(frames)} кадров в {output_dir}")


# Использование
if __name__ == "__main__":
    preprocessor = VideoPreprocessor(r"25_2-10.mp4")
    frames = preprocessor.process()
    preprocessor.save_frames(frames, "extracted_frames")
    
    print(f"\nСтатистика:")
    print(f"  Всего кадров: {len(frames)}")
    print(f"  Средняя чёткость: {np.mean([s for _,_,s in frames]):.1f}")
    print(f"  Диапазон: {min([s for _,_,s in frames]):.1f} – {max([s for _,_,s in frames]):.1f}")

In [1]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from tqdm import tqdm
import os

# Настройки
VIDEO_PATH = "data/43_15.mp4"
MODEL_PATH = "best.pt"
CROPS_DIR = "crops"
os.makedirs(CROPS_DIR, exist_ok=True)
CONF_THRESH = 0.05

print(" Загрузка модели...")
model = YOLO(MODEL_PATH, verbose=False)

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
saved = 0

print(f" Обработка видео ({total_frames} кадров)...")
for idx in tqdm(range(total_frames), desc="Инференс"):
    ret, frame = cap.read()
    if not ret: break
    
    # Поворот 90° против часовой (ТЗ)
    frame = cv2.rotate(frame, cv2.ROTATE_90_COUNTERCLOCKWISE)
    ts_ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))
    
    # Детекция
    results = model(frame, conf=CONF_THRESH, verbose=False)
    
    for i, box in enumerate(results[0].boxes):
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        
        # Базовый фильтр геометрии (убираем явный мусор)
        bw, bh = x2-x1, y2-y1
        if bw*bh < 2000: continue
        if not (1.0 < bw/max(bh,1) < 5.0): continue
        
        # Вырезаем и сохраняем кроп
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0: continue
        
        fname = f"frame_{idx:05d}_ts{ts_ms}_bbox_{x1}_{y1}_{x2}_{y2}.jpg"
        cv2.imwrite(os.path.join(CROPS_DIR, fname), crop)
        saved += 1

cap.release()
print(f" Сохранено {saved} кропов в папке {CROPS_DIR}/")

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\kolua\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
 Загрузка модели...
 Обработка видео (299 кадров)...


Инференс: 100%|██████████| 299/299 [00:27<00:00, 10.69it/s]


 Сохранено 3374 кропов в папке crops/


In [8]:
from pathlib import Path
import csv
import re
import shutil

import cv2
import easyocr
from tqdm import tqdm

CROPS_DIR = Path("crops")
OUT_CSV = Path("crops_result_first10_better.csv")
DEBUG_DIR = Path("debug_price_zones")
FIRST_N = 10

FORCE_99_KOP_IF_MISSING = True

if DEBUG_DIR.exists():
    shutil.rmtree(DEBUG_DIR)

DEBUG_DIR.mkdir(parents=True, exist_ok=True)

reader = easyocr.Reader(["en"], gpu=False)

def clean_digits(text: str) -> str:
    return re.sub(r"\D+", "", text)

def get_price_zone(img):
    h, w = img.shape[:2]

    y1 = int(h * 0.25)
    y2 = int(h * 1.00)
    x1 = int(w * 0.00)
    x2 = int(w * 1.00)

    return img[y1:y2, x1:x2]

def rotate_image(img, angle):
    h, w = img.shape[:2]
    center = (w // 2, h // 2)

    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)

    return cv2.warpAffine(
        img,
        matrix,
        (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255
    )

def preprocess_variants(price_zone):
    gray = cv2.cvtColor(price_zone, cv2.COLOR_BGR2GRAY)

    gray = cv2.resize(
        gray,
        None,
        fx=4,
        fy=4,
        interpolation=cv2.INTER_CUBIC
    )

    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_clahe = clahe.apply(gray)

    _, otsu = cv2.threshold(
        gray_clahe,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    adaptive = cv2.adaptiveThreshold(
        gray_clahe,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        41,
        7
    )

    base_variants = {
        "gray": gray_clahe,
        "otsu": otsu,
        "adaptive": adaptive,
    }

    variants = {}

    for name, img in base_variants.items():
        variants[f"{name}_rot0"] = img
        variants[f"{name}_rot_minus4"] = rotate_image(img, -4)
        variants[f"{name}_rot_plus4"] = rotate_image(img, 4)

    return variants

def bbox_stats(box):
    xs = [p[0] for p in box]
    ys = [p[1] for p in box]

    x1 = min(xs)
    x2 = max(xs)
    y1 = min(ys)
    y2 = max(ys)

    width = x2 - x1
    height = y2 - y1
    area = width * height

    return {
        "x1": float(x1),
        "x2": float(x2),
        "y1": float(y1),
        "y2": float(y2),
        "xc": float((x1 + x2) / 2),
        "yc": float((y1 + y2) / 2),
        "w": float(width),
        "h": float(height),
        "area": float(area),
    }

def read_candidates(variants):
    candidates = []

    for variant_name, img in variants.items():
        result = reader.readtext(
            img,
            detail=1,
            paragraph=False,
            allowlist="0123456789",
            text_threshold=0.25,
            low_text=0.15,
            link_threshold=0.2
        )

        img_h, img_w = img.shape[:2]

        for box, text, conf in result:
            digits = clean_digits(text)

            if not digits:
                continue

            stats = bbox_stats(box)

            candidates.append({
                "variant": variant_name,
                "text": text,
                "digits": digits,
                "conf": float(conf),
                "img_w": img_w,
                "img_h": img_h,
                **stats,
            })

    return candidates

def prepare_rub_candidate(c):
    digits = c["digits"]

    if digits == "99":
        return None

    if len(digits) > 6:
        return None

    if len(digits) >= 5 and digits.endswith("99"):
        rub_digits = digits[:-2]
        kop_digits = "99"
    else:
        rub_digits = digits
        kop_digits = ""

    if not rub_digits:
        return None

    value = int(rub_digits)

    if value < 10 or value > 9999:
        return None

    img_w = c["img_w"]
    img_h = c["img_h"]

    if c["yc"] < img_h * 0.28:
        return None

    if c["xc"] < img_w * 0.12:
        return None

    if c["h"] < img_h * 0.10:
        return None

    item = c.copy()
    item["rub_digits"] = rub_digits
    item["kop_digits"] = kop_digits

    item["score"] = (
        item["h"] * 5
        + item["area"] * 0.01
        + item["conf"] * 80
    )

    return item

def find_kop(candidates, rub):
    if rub.get("kop_digits") == "99":
        return "99"

    kop_candidates = []

    for c in candidates:
        if c["digits"] != "99":
            continue

        if c["variant"] != rub["variant"]:
            continue

        is_right_side = c["xc"] > rub["xc"]
        is_near_price = (
            rub["y1"] - rub["h"] * 0.9
            <= c["yc"]
            <= rub["y2"] + rub["h"] * 0.4
        )

        if is_right_side and is_near_price:
            kop_candidates.append(c)

    if kop_candidates:
        kop_candidates.sort(
            key=lambda c: (
                abs(c["yc"] - rub["yc"]),
                -c["conf"]
            )
        )
        return "99"

    if FORCE_99_KOP_IF_MISSING:
        return "99"

    return ""

def select_price(candidates):
    rub_candidates = []

    for c in candidates:
        prepared = prepare_rub_candidate(c)

        if prepared:
            rub_candidates.append(prepared)

    if not rub_candidates:
        return "", "", "", ""

    rub_candidates.sort(key=lambda c: c["score"], reverse=True)

    rub = rub_candidates[0]
    kop = find_kop(candidates, rub)

    rub_digits = str(int(rub["rub_digits"]))

    if kop:
        price = f"{rub_digits}.{kop}"
    else:
        price = rub_digits

    return price, rub_digits, kop, rub["variant"]

def recognize_price(image_path: Path):
    img = cv2.imread(str(image_path))

    if img is None:
        return "", "", "", "", "", "cannot read image"

    price_zone = get_price_zone(img)
    cv2.imwrite(str(DEBUG_DIR / f"zone_{image_path.name}"), price_zone)

    variants = preprocess_variants(price_zone)

    for variant_name, variant_img in variants.items():
        cv2.imwrite(
            str(DEBUG_DIR / f"{variant_name}_{image_path.name}"),
            variant_img
        )

    candidates = read_candidates(variants)
    price, rub, kop, best_variant = select_price(candidates)

    candidates.sort(
        key=lambda c: (c["area"], c["conf"]),
        reverse=True
    )

    raw_ocr = " | ".join(
        f'{c["digits"]} variant={c["variant"]} conf={round(c["conf"], 3)} area={round(c["area"])}'
        for c in candidates[:20]
    )

    return price, rub, kop, best_variant, raw_ocr, ""

all_image_paths = sorted(
    list(CROPS_DIR.glob("*.jpg")) +
    list(CROPS_DIR.glob("*.jpeg")) +
    list(CROPS_DIR.glob("*.png"))
)

image_paths = all_image_paths[:FIRST_N] if FIRST_N else all_image_paths

print("Текущая папка:", Path.cwd())
print("Папка crops:", CROPS_DIR.resolve())
print("Всего картинок в crops:", len(all_image_paths))
print("Будет обработано:", len(image_paths))

for p in image_paths:
    print(p.name)

rows = []

for image_path in tqdm(image_paths):
    try:
        price, rub, kop, best_variant, raw_ocr, error = recognize_price(image_path)

        rows.append({
            "file": image_path.name,
            "current_price": price,
            "rub": rub,
            "kop": kop,
            "best_variant": best_variant,
            "raw_ocr": raw_ocr,
            "error": error,
        })

    except Exception as e:
        rows.append({
            "file": image_path.name,
            "current_price": "",
            "rub": "",
            "kop": "",
            "best_variant": "",
            "raw_ocr": "",
            "error": str(e),
        })

with OUT_CSV.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "file",
            "current_price",
            "rub",
            "kop",
            "best_variant",
            "raw_ocr",
            "error",
        ]
    )
    writer.writeheader()
    writer.writerows(rows)

print()
print(f"CSV сохранён: {OUT_CSV}")
print(f"Обработано картинок: {len(rows)}")
print(f"Debug сохранён в: {DEBUG_DIR}")

Using CPU. Note: This module is much faster with a GPU.


Текущая папка: e:\Work_2\Фриланс\полка csv\Lenta_Tech_detector
Папка crops: E:\Work_2\Фриланс\полка csv\Lenta_Tech_detector\crops
Всего картинок в crops: 3374
Будет обработано: 10
frame_00000_ts0_bbox_1257_1368_1506_1599.jpg
frame_00000_ts0_bbox_1425_2543_1637_2730.jpg
frame_00000_ts0_bbox_1695_2507_1894_2689.jpg
frame_00000_ts0_bbox_372_1357_621_1567.jpg
frame_00000_ts0_bbox_509_3401_698_3540.jpg
frame_00000_ts0_bbox_699_2543_912_2749.jpg
frame_00000_ts0_bbox_82_2529_307_2711.jpg
frame_00000_ts0_bbox_919_3417_1102_3546.jpg
frame_00000_ts0_bbox_951_2548_1180_2738.jpg
frame_00001_ts50_bbox_1257_1368_1506_1600.jpg


100%|██████████| 10/10 [04:44<00:00, 28.45s/it]


CSV сохранён: crops_result_first10_better.csv
Обработано картинок: 10
Debug сохранён в: debug_price_zones


In [9]:
from pathlib import Path
import csv
import math
import re

INPUT_CSV = Path("crops_result_first10_better.csv")
OUT_CSV = Path("crops_result_first10_fixed.csv")

def parse_raw_ocr(raw_ocr: str):
    result = []

    for part in str(raw_ocr).split(" | "):
        match = re.match(
            r"(\d+)\s+variant=([^ ]+)\s+conf=([0-9.]+)\s+area=([0-9.]+)",
            part
        )

        if not match:
            continue

        result.append({
            "digits": match.group(1),
            "variant": match.group(2),
            "conf": float(match.group(3)),
            "area": float(match.group(4)),
        })

    return result

def normalize_price_digits(digits: str):
    digits = digits.lstrip("0") or "0"

    if digits == "99":
        return None

    if len(digits) == 3:
        return digits

    if len(digits) == 4:
        if digits[0] == digits[1]:
            return digits[1:]

        return digits[:3]

    return None

def fix_price_from_raw_ocr(raw_ocr: str):
    groups = {}

    for item in parse_raw_ocr(raw_ocr):
        if not item["variant"].startswith("gray"):
            continue

        rub = normalize_price_digits(item["digits"])

        if rub is None:
            continue

        value = int(rub)

        if value < 10 or value > 9999:
            continue

        score = math.sqrt(item["area"]) * (0.5 + item["conf"])
        groups[rub] = groups.get(rub, 0) + score + 20

    if not groups:
        return ""

    best_rub = max(groups, key=groups.get)

    return f"{int(best_rub)}.99"

rows = []

with INPUT_CSV.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        row["current_price_old"] = row.get("current_price", "")
        row["current_price"] = fix_price_from_raw_ocr(row.get("raw_ocr", ""))
        rows.append(row)

fieldnames = [
    "file",
    "current_price",
    "current_price_old",
    "rub",
    "kop",
    "best_variant",
    "raw_ocr",
    "error",
]

with OUT_CSV.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    for row in rows:
        writer.writerow({name: row.get(name, "") for name in fieldnames})

print(f"CSV сохранён: {OUT_CSV}")
print(f"Обработано строк: {len(rows)}")

CSV сохранён: crops_result_first10_fixed.csv
Обработано строк: 10
